# 8A Targeted improvement experiments (issue #24)

Builds on `error_analysis.ipynb` (PR #29), whose best model is `ensemble_top2` (the mean probability of
`modernbert_v2_biasinit` + `modernbert_v1_6ep`) with per-category best-F1 cutoffs picked on tune
(validation macro F1 0.559). Each experiment here changes **one** thing relative to its parent:

| ID | Change | Parent | Retraining? |
|---|---|---|---|
| E1 | Threshold floor: categories with fewer than N tune positives keep the 0.5 cutoff | `ensemble_top2`, best-F1 cutoffs | no |
| E2 | Learning rate 3e-5 instead of 5e-5 | `modernbert_v2_biasinit` | yes (Colab) |
| E3 | Class weights (`--pos-weight sqrt`) on top of bias init | `modernbert_v2_biasinit` | yes (Colab) |

Same rules as before: every choice is made on **tune**, **validation** is only used to report, `test.json`
is never touched, and all **41** categories are predicted.

**Before running:** `data/processed/` must come from the current `main_pipeline.ipynb` (8549 chunks). Run from
`notebooks/`. Outputs go to `results/improvement_experiments/`.

## Setup

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

from eval_metrics import multilabel_metrics
from splits import SPLIT_NAMES, load_splits

REPO_DIR = Path(os.getcwd()).parent
PROCESSED_DIR = REPO_DIR / "data" / "processed"
RESULTS_DIR = REPO_DIR / "results"
OUT_DIR = RESULTS_DIR / "improvement_experiments"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_N_CHUNKS = 8549  # same check as modernbert_finetune.py and error_analysis.ipynb

chunks_df = pd.read_parquet(PROCESSED_DIR / "chunks_df.parquet")
chunk_labels_df = pd.read_parquet(PROCESSED_DIR / "chunk_labels_df.parquet")
if len(chunks_df) != EXPECTED_N_CHUNKS:
    raise ValueError(f"{PROCESSED_DIR} has {len(chunks_df)} chunks, expected {EXPECTED_N_CHUNKS}: stale data "
                     "from the old cl100k chunker, re-run main_pipeline.ipynb first")
chunks_df = chunks_df[chunks_df["chunk_start"] >= 0]

# same chunk table as error_analysis.ipynb (without the text/position columns it only needs for 7B)
label_matrix = chunk_labels_df.pivot_table(
    index="chunk_id", columns="category", values="label", aggfunc="max", fill_value=0
).astype(int)
categories = label_matrix.columns.tolist()
assert len(categories) == 41, "project rules say predict all 41 CUAD categories"

table = chunks_df.set_index("chunk_id")[["contract_id"]].join(label_matrix, how="inner").reset_index()

splits = load_splits()
parts = {name: table[table["contract_id"].isin(set(splits[name]))].reset_index(drop=True) for name in SPLIT_NAMES}
y = {name: parts[name][categories].to_numpy() for name in SPLIT_NAMES}
contract_sets = [set(splits[name]) for name in SPLIT_NAMES]
assert all(not (a & b) for i, a in enumerate(contract_sets) for b in contract_sets[i + 1:]), "a contract is in two splits"
for name in SPLIT_NAMES:
    print(f"{name:>5}: {len(parts[name]):5d} chunks, {parts[name]['contract_id'].nunique():3d} contracts, "
          f"{int(y[name].sum()):5d} positive pairs")

train:  5422 chunks, 260 contracts,  5318 positive pairs
 tune:  1286 chunks,  66 contracts,  1308 positive pairs
  val:  1841 chunks,  82 contracts,  1691 positive pairs


In [2]:
def load_probs(run, split):
    """A run's saved probabilities, re-ordered to line up row for row with parts[split] (from error_analysis.ipynb)."""
    saved = pd.read_parquet(RESULTS_DIR / run / f"{split}_probs.parquet").set_index("chunk_id")
    mismatch = set(parts[split]["chunk_id"]) ^ set(saved.index)
    if mismatch:
        raise ValueError(f"{run}/{split}: {len(mismatch)} chunk ids differ from data/processed: stale data?")
    return saved.loc[parts[split]["chunk_id"], categories].to_numpy()


# the current best model, pinned by name: error_analysis.ipynb picks its top 2 by tune macro AP over every
# results/modernbert_* folder, so adding the E2/E3 runs could otherwise silently change E1's parent
PARENT_RUNS = ["modernbert_v2_biasinit", "modernbert_v1_6ep"]
probs = {run: {split: load_probs(run, split) for split in ("tune", "val")} for run in PARENT_RUNS}
probs["ensemble_top2"] = {split: np.mean([probs[run][split] for run in PARENT_RUNS], axis=0)
                          for split in ("tune", "val")}
print(f"ensemble_top2 = {' + '.join(PARENT_RUNS)}")

ensemble_top2 = modernbert_v2_biasinit + modernbert_v1_6ep


## Threshold helpers

`pick_thresholds` is `error_analysis.ipynb`'s best-F-beta cutoff picker with one change, the variable E1 tests:
`min_positives`. A category with fewer than `min_positives` positives in the fitting set keeps the 0.5 default
instead of getting a tuned cutoff. `min_positives=1` is exactly the old rule (only categories with 0 positives keep
0.5), so the parent is reproduced first as a check.

In [3]:
DEFAULT_THRESHOLD = 0.5
CANDIDATE_THRESHOLDS = np.round(np.arange(0.01, 0.96, 0.01), 2)  # same grid as error_analysis.ipynb


def pick_thresholds(y_true, scores, beta=1, min_positives=1, n_pos_for_floor=None):
    """One cutoff per category: the candidate with the best F-beta on (y_true, scores), ties to the lowest cutoff.
    Categories with fewer than `min_positives` positives keep the default. `n_pos_for_floor` overrides which counts
    the floor looks at (used in cross-validation, so a fold floors the same categories the full tune set would)."""
    n_pos_all = y_true.sum(axis=0) if n_pos_for_floor is None else np.asarray(n_pos_for_floor)
    picked = np.full(len(categories), DEFAULT_THRESHOLD)
    b2 = beta ** 2
    for i in range(len(categories)):
        n_pos = y_true[:, i].sum()
        if n_pos == 0 or n_pos_all[i] < min_positives:
            continue
        pred = scores[:, i, None] >= CANDIDATE_THRESHOLDS        # (n_chunks x n_candidates)
        tp = (pred & (y_true[:, i, None] == 1)).sum(axis=0)
        fp = pred.sum(axis=0) - tp
        fn = n_pos - tp
        fbeta = (1 + b2) * tp / np.maximum((1 + b2) * tp + b2 * fn + fp, 1)
        picked[i] = CANDIDATE_THRESHOLDS[int(np.argmax(fbeta))]
    return picked


def log_row(experiment_id, parent, model, what, thr):
    """One experiment-log row, same metric columns as results/error_analysis/experiment_log.csv."""
    tune_summary, _ = multilabel_metrics(y["tune"], probs[model]["tune"], categories, thr)
    val_summary, _ = multilabel_metrics(y["val"], probs[model]["val"], categories, thr)
    return {
        "experiment_id": experiment_id,
        "parent": parent,
        "model": model,
        "what": what,
        "tune_macro_ap": tune_summary["macro_ap_present"],
        "tune_macro_f1": tune_summary["macro_f1_present"],
        "val_macro_ap": val_summary["macro_ap_present"],
        "val_micro_precision": val_summary["micro_precision"],
        "val_micro_recall": val_summary["micro_recall"],
        "val_micro_f1": val_summary["micro_f1"],
        "val_macro_f1": val_summary["macro_f1_present"],
    }


# sanity check: min_positives=1 must reproduce error_analysis.ipynb's best model exactly
parent_thr = pick_thresholds(y["tune"], probs["ensemble_top2"]["tune"])
logged = pd.read_csv(RESULTS_DIR / "error_analysis" / "experiment_log.csv")
logged = logged[(logged["model"] == "ensemble_top2") & (logged["thresholds"] == "best F1")].iloc[0]
parent_row = log_row("E0", "", "ensemble_top2", "current best: ensemble_top2, best-F1 cutoffs", parent_thr)
for col in ("val_macro_f1", "val_micro_f1", "val_macro_ap"):
    assert np.isclose(parent_row[col], logged[col]), f"{col}: {parent_row[col]:.4f} vs logged {logged[col]:.4f}"
print(f"parent reproduced: val macro F1 {parent_row['val_macro_f1']:.3f}, micro F1 {parent_row['val_micro_f1']:.3f}, "
      f"macro AP {parent_row['val_macro_ap']:.3f}")

parent reproduced: val macro F1 0.559, micro F1 0.664, macro AP 0.593


## E1 Threshold floor

**Why:** 7B found that most false alarms come from one category. Affiliate License-Licensor has only 2 positives
in tune, so its best-F1 cutoff landed at 0.01. That makes it fire on License Grant / Non-Transferable License chunks
(~170 FPs in validation). A cutoff fit on a handful of examples is mostly noise. The fix is to tune a cutoff only
when there's enough evidence, and otherwise fall back to 0.5.

**How N is chosen: cross-validation on tune.** Picking N by plain tune F1 would always choose "no floor", because
cutoffs fit on tune always score best on tune. Instead, the 66 tune contracts are split into 5 folds (by contract).
For each fold, cutoffs are fit on the other 4 and applied to the held-out fold. The held-out predictions from all 5
folds are pooled into one out-of-fold prediction for every tune chunk, and N is chosen by macro F1 (present
categories) on that. The floor counts positives on the **full** tune set, so each fold floors exactly the
categories the final rule will. Ties go to the smallest N (the least change from the parent). Validation is not
looked at until N is fixed.

In [4]:
FLOOR_GRID = [1, 3, 5, 10, 15, 20]  # 1 = parent rule (only categories with 0 positives keep 0.5)
N_FOLDS = 5

tune_scores = probs["ensemble_top2"]["tune"]
tune_n_pos = y["tune"].sum(axis=0)
folds = list(GroupKFold(n_splits=N_FOLDS).split(tune_scores, groups=parts["tune"]["contract_id"]))

sweep = []
for floor in FLOOR_GRID:
    oof_pred = np.zeros_like(y["tune"])
    for fit_idx, held_idx in folds:
        thr = pick_thresholds(y["tune"][fit_idx], tune_scores[fit_idx], min_positives=floor, n_pos_for_floor=tune_n_pos)
        oof_pred[held_idx] = tune_scores[held_idx] >= thr
    # oof_pred is already 0/1, so scoring it at 0.5 just reads it back
    cv_summary, _ = multilabel_metrics(y["tune"], oof_pred, categories)
    sweep.append({
        "min_positives": floor,
        "n_categories_floored": int(((tune_n_pos > 0) & (tune_n_pos < floor)).sum()),
        "cv_tune_macro_f1": cv_summary["macro_f1_present"],
        "cv_tune_micro_f1": cv_summary["micro_f1"],
    })
sweep_df = pd.DataFrame(sweep)
best_floor = int(sweep_df.loc[sweep_df["cv_tune_macro_f1"].idxmax(), "min_positives"])  # idxmax: first = smallest N
print(f"chosen on tune (cross-validated): min_positives = {best_floor}\n")
print(sweep_df.round(3).to_string(index=False))

chosen on tune (cross-validated): min_positives = 1

 min_positives  n_categories_floored  cv_tune_macro_f1  cv_tune_micro_f1
             1                     0             0.540             0.657
             3                     2             0.538             0.688
             5                     5             0.529             0.695
            10                     9             0.530             0.703
            15                    14             0.503             0.706
            20                    15             0.495             0.702


### E1 on validation (report only)

Final cutoffs are fit on the full tune set with the chosen floor, then scored once on validation against the parent.
The whole sweep is also shown on validation for transparency, but it played no part in choosing N.

In [5]:
e1_thr = pick_thresholds(y["tune"], tune_scores, min_positives=best_floor)
e1_row = log_row("E1", "E0", "ensemble_top2",
                 f"threshold floor: categories with < {best_floor} tune positives keep 0.5 (N picked by 5-fold CV on tune)",
                 e1_thr)
compare = pd.DataFrame([parent_row, e1_row]).set_index("experiment_id")
print(compare.drop(columns=["parent", "model", "what"]).T.round(3).to_string(), "\n")

# report-only: every floor on validation
val_sweep = []
for floor in FLOOR_GRID:
    s, _ = multilabel_metrics(y["val"], probs["ensemble_top2"]["val"], categories,
                              pick_thresholds(y["tune"], tune_scores, min_positives=floor))
    val_sweep.append({"min_positives": floor, "val_macro_f1": s["macro_f1_present"], "val_micro_f1": s["micro_f1"],
                      "val_micro_precision": s["micro_precision"], "val_micro_recall": s["micro_recall"]})
sweep_df = sweep_df.merge(pd.DataFrame(val_sweep), on="min_positives")
sweep_df.to_csv(OUT_DIR / "e1_floor_sweep.csv", index=False)
print("full sweep (val columns are report-only):")
print(sweep_df.round(3).to_string(index=False))

experiment_id           E0     E1
tune_macro_ap        0.590  0.590
tune_macro_f1        0.604  0.604
val_macro_ap         0.593  0.593
val_micro_precision  0.633  0.633
val_micro_recall     0.698  0.698
val_micro_f1         0.664  0.664
val_macro_f1         0.559  0.559 



full sweep (val columns are report-only):
 min_positives  n_categories_floored  cv_tune_macro_f1  cv_tune_micro_f1  val_macro_f1  val_micro_f1  val_micro_precision  val_micro_recall
             1                     0             0.540             0.657         0.559         0.664                0.633             0.698
             3                     2             0.538             0.688         0.559         0.696                0.699             0.694
             5                     5             0.529             0.695         0.545         0.700                0.710             0.691
            10                     9             0.530             0.703         0.545         0.703                0.720             0.687
            15                    14             0.503             0.706         0.522         0.712                0.761             0.669
            20                    15             0.495             0.702         0.526         0.715                0.

**Result:** cross-validated tune macro F1 is highest with no floor (N = 1), so E1 as selected is identical to
its parent. On the metric chosen up front, the floor **didn't help**.

**Noted alternative, N = 3 (seen after validation, not selected):** it floors only the 2 categories with 2 tune
positives, which keep 0.5 instead of a tuned cutoff:

- **Affiliate License-Licensor** (2 tune / 7 val positives, tuned cutoff 0.01): 137 validation FPs drop to 5, but recall falls from 7/7 to 1/7
- **Most Favored Nation** (2 tune / 7 val positives, tuned cutoff 0.02): 46 validation FPs drop to 0, recall 1/7 -> 0/7

Together that's 684 -> 506 validation false positives (-178).

Macro F1 is flat (CV tune 0.538 vs 0.540; validation 0.559 both), while micro F1 and precision rise
(validation micro F1 0.664 -> 0.696, precision 0.633 -> 0.699). It's a precision option for the team to weigh, not
a tune-selected result. The table below shows what it changes per category (report-only).

In [6]:
# per category, report-only: what the noted N = 3 floor changes vs. the parent
ALT_FLOOR = 3
alt_thr = pick_thresholds(y["tune"], tune_scores, min_positives=ALT_FLOOR)
val_scores = probs["ensemble_top2"]["val"]
y_val = y["val"]
changed = np.flatnonzero(~np.isclose(parent_thr, alt_thr))
_, parent_cat = multilabel_metrics(y_val, val_scores, categories, parent_thr)
_, alt_cat = multilabel_metrics(y_val, val_scores, categories, alt_thr)


def fp_count(thr):
    return ((val_scores >= thr) & (y_val == 0)).sum(axis=0)


floored_df = pd.DataFrame({
    "category": np.array(categories)[changed],
    "n_pos_tune": tune_n_pos[changed],
    "n_pos_val": y_val.sum(axis=0)[changed],
    "cutoff_parent": parent_thr[changed],
    f"cutoff_n{ALT_FLOOR}": alt_thr[changed],
    "fp_parent": fp_count(parent_thr)[changed],
    f"fp_n{ALT_FLOOR}": fp_count(alt_thr)[changed],
    "recall_parent": parent_cat["recall"].to_numpy()[changed],
    f"recall_n{ALT_FLOOR}": alt_cat["recall"].to_numpy()[changed],
    "f1_parent": parent_cat["f1"].to_numpy()[changed],
    f"f1_n{ALT_FLOOR}": alt_cat["f1"].to_numpy()[changed],
}).sort_values("n_pos_tune")
floored_df.to_csv(OUT_DIR / f"e1_floored_categories_n{ALT_FLOOR}.csv", index=False)
pd.DataFrame({"category": categories, "threshold_parent": parent_thr, "threshold_e1": e1_thr,
              f"threshold_n{ALT_FLOOR}": alt_thr}).to_csv(OUT_DIR / "e1_selected_thresholds.csv", index=False)

total_fp_parent, total_fp_alt = int(fp_count(parent_thr).sum()), int(fp_count(alt_thr).sum())
print(f"N = {ALT_FLOOR} (report-only): validation false positives {total_fp_parent} -> {total_fp_alt} "
      f"({total_fp_alt - total_fp_parent:+d}), from flooring {len(changed)} categories: "
      f"{', '.join(floored_df['category'])}\n")
print(floored_df.round(2).to_string(index=False))

N = 3 (report-only): validation false positives 684 -> 506 (-178), from flooring 2 categories: Affiliate License-Licensor, Most Favored Nation

                  category  n_pos_tune  n_pos_val  cutoff_parent  cutoff_n3  fp_parent  fp_n3  recall_parent  recall_n3  f1_parent  f1_n3
Affiliate License-Licensor           2          7           0.01        0.5        137      5           1.00       0.14       0.09   0.15
       Most Favored Nation           2          7           0.02        0.5         46      0           0.14       0.00       0.04   0.00


## E2 / E3 Retraining experiments

Two new ModernBERT runs, each changing one training setting relative to `modernbert_v2_biasinit` (the best single
run), trained with `modernbert_finetune.py` in Colab and saved to `results/<run>/` like the earlier runs:

- **E2** `modernbert_v5_biasinit_lr3e-5`: learning rate 3e-5 instead of 5e-5.
- **E3** `modernbert_v6_biasinit_posweight`: adds class weights (`--pos-weight sqrt`) on top of bias init. #28 tried
  class weights only on plain v1 (v3) and skipped this combination.

(The runs were named `..._baisinit_...` in Colab. The folders and `run_name` fields were corrected afterwards, but the
Google Drive paths in `run_config.json` / `metrics.json` keep the original spelling, because that's where the models are.)

**Same setup as v2 except the one setting:** A100 GPU, bf16, seed 0, torch 2.11.0, 3 epochs, effective batch 16, and the
same training script (unchanged since v2's commit). The one other difference is **transformers 5.21.0 vs 5.20.1** for v2,
because Colab updated it. No control rerun of v2 was done, so a small part of any difference could come from that.
The cell below prints the versions straight from each `run_config.json`.

In [7]:
PARENT_RUN = "modernbert_v2_biasinit"
NEW_RUNS = {
    "E2": ("modernbert_v5_biasinit_lr3e-5", "v2 with learning rate 3e-5 instead of 5e-5"),
    "E3": ("modernbert_v6_biasinit_posweight", "v2 with class weights (pos_weight=sqrt) added"),
}
for run, _ in NEW_RUNS.values():
    probs[run] = {split: load_probs(run, split) for split in ("tune", "val")}

# integrity check: scores recomputed from the saved probabilities must match what training wrote to metrics.json
for run in [PARENT_RUN, *(r for r, _ in NEW_RUNS.values())]:
    saved = json.loads((RESULTS_DIR / run / "metrics.json").read_text())
    for split in ("tune", "val"):
        summary, _ = multilabel_metrics(y[split], probs[run][split], categories)
        for key in ("micro_ap", "macro_ap_present", "micro_f1", "macro_f1_present"):
            assert np.isclose(summary[key], saved[split][key]), f"{run}/{split}/{key}: {summary[key]} vs {saved[split][key]}"

setup = {}
for run in [PARENT_RUN, *(r for r, _ in NEW_RUNS.values())]:
    cfg = json.loads((RESULTS_DIR / run / "run_config.json").read_text())
    setup[run] = {
        "learning_rate": cfg["options"]["learning_rate"], "bias_init": cfg["options"]["bias_init"],
        "pos_weight": cfg["options"]["pos_weight"], "epochs": cfg["options"]["epochs"],
        "device": cfg["device"], "precision": cfg["training"]["precision"], "seed": cfg["training"]["seed"],
        "torch": cfg["versions"]["torch"], "transformers": cfg["versions"]["transformers"],
        "git_commit": cfg["git"]["commit"][:7],
    }
print("saved probabilities match every metrics.json\n")
print(pd.DataFrame(setup).to_string())

saved probabilities match every metrics.json

              modernbert_v2_biasinit modernbert_v5_biasinit_lr3e-5 modernbert_v6_biasinit_posweight
learning_rate                0.00005                       0.00003                          0.00005
bias_init                       True                          True                             True
pos_weight                      None                          None                             sqrt
epochs                             3                             3                                3
device         NVIDIA A100-SXM4-40GB         NVIDIA A100-SXM4-40GB            NVIDIA A100-SXM4-40GB
precision                       bf16                          bf16                             bf16
seed                               0                             0                                0
torch                   2.11.0+cu130                  2.11.0+cu130                     2.11.0+cu130
transformers                  5.20.1                  

In [8]:
# each run under the same two cutoff rules as error_analysis.ipynb: 0.5, and per-category best-F1 picked on tune
STRATEGIES = {"0.5 (default)": None, "best F1": 1}
compare_rows = []
for run in [PARENT_RUN, *(r for r, _ in NEW_RUNS.values())]:
    for strategy, beta in STRATEGIES.items():
        thr = np.full(len(categories), DEFAULT_THRESHOLD) if beta is None else pick_thresholds(y["tune"], probs[run]["tune"], beta)
        tune_summary, _ = multilabel_metrics(y["tune"], probs[run]["tune"], categories, thr)
        val_summary, _ = multilabel_metrics(y["val"], probs[run]["val"], categories, thr)
        compare_rows.append({
            "model": run, "thresholds": strategy,
            "tune_macro_ap": tune_summary["macro_ap_present"],
            "val_micro_ap": val_summary["micro_ap"], "val_macro_ap": val_summary["macro_ap_present"],
            "val_micro_precision": val_summary["micro_precision"], "val_micro_recall": val_summary["micro_recall"],
            "val_micro_f1": val_summary["micro_f1"], "val_macro_f1": val_summary["macro_f1_present"],
        })
retrain_df = pd.DataFrame(compare_rows)
retrain_df.to_csv(OUT_DIR / "e2_e3_vs_v2.csv", index=False)
print("tune_macro_ap is what a choice between runs would be made on; val_* is report-only\n")
print(retrain_df.round(3).to_string(index=False))

tune_macro_ap is what a choice between runs would be made on; val_* is report-only

                           model    thresholds  tune_macro_ap  val_micro_ap  val_macro_ap  val_micro_precision  val_micro_recall  val_micro_f1  val_macro_f1
          modernbert_v2_biasinit 0.5 (default)          0.569         0.761         0.584                0.826             0.608         0.700         0.506
          modernbert_v2_biasinit       best F1          0.569         0.761         0.584                0.594             0.693         0.640         0.536
   modernbert_v5_biasinit_lr3e-5 0.5 (default)          0.508         0.725         0.528                0.814             0.547         0.654         0.424
   modernbert_v5_biasinit_lr3e-5       best F1          0.508         0.725         0.528                0.555             0.672         0.608         0.492
modernbert_v6_biasinit_posweight 0.5 (default)          0.570         0.744         0.587                0.640             0.729   

### How sure are the differences?

Same check as `results/README.md`: resample the 82 validation contracts with replacement 1,000 times and recompute
each run's AP minus v2's. This covers which contracts happen to be in validation, not training randomness: each
setup was trained once, with one seed.

In [9]:
from sklearn.metrics import average_precision_score

N_BOOT = 1000
val_contracts = parts["val"]["contract_id"].to_numpy()
contract_rows = {c: np.flatnonzero(val_contracts == c) for c in np.unique(val_contracts)}
contract_ids = np.array(list(contract_rows))


def ap_scores(y_true, scores):
    """(micro AP, macro AP over categories with positives), same definitions as eval_metrics."""
    present = np.flatnonzero(y_true.sum(axis=0) > 0)
    macro = np.mean([average_precision_score(y_true[:, i], scores[:, i]) for i in present])
    return average_precision_score(y_true.ravel(), scores.ravel()), macro


rng = np.random.default_rng(0)
boot = {run: [] for run, _ in NEW_RUNS.values()}
for _ in range(N_BOOT):
    rows = np.concatenate([contract_rows[c] for c in rng.choice(contract_ids, len(contract_ids))])
    parent_micro, parent_macro = ap_scores(y["val"][rows], probs[PARENT_RUN]["val"][rows])
    for run in boot:
        micro, macro = ap_scores(y["val"][rows], probs[run]["val"][rows])
        boot[run].append((micro - parent_micro, macro - parent_macro))

ci_rows = []
for exp_id, (run, _) in NEW_RUNS.items():
    point_micro = ap_scores(y["val"], probs[run]["val"])[0] - ap_scores(y["val"], probs[PARENT_RUN]["val"])[0]
    point_macro = ap_scores(y["val"], probs[run]["val"])[1] - ap_scores(y["val"], probs[PARENT_RUN]["val"])[1]
    d = np.array(boot[run])
    ci_rows.append({
        "comparison": f"{exp_id} - v2",
        "micro_ap_diff": point_micro, "micro_ap_95_low": np.percentile(d[:, 0], 2.5), "micro_ap_95_high": np.percentile(d[:, 0], 97.5),
        "macro_ap_diff": point_macro, "macro_ap_95_low": np.percentile(d[:, 1], 2.5), "macro_ap_95_high": np.percentile(d[:, 1], 97.5),
    })
ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(OUT_DIR / "e2_e3_bootstrap.csv", index=False)
print(ci_df.round(3).to_string(index=False))

comparison  micro_ap_diff  micro_ap_95_low  micro_ap_95_high  macro_ap_diff  macro_ap_95_low  macro_ap_95_high
   E2 - v2         -0.036           -0.044            -0.026         -0.056           -0.081            -0.037
   E3 - v2         -0.017           -0.026            -0.009          0.002           -0.023             0.029


In [10]:
# rare categories: the 9 with < 50 training positives and some validation positives (as in results/README.md)
train_n_pos = y["train"].sum(axis=0)
val_n_pos = y["val"].sum(axis=0)
rare = np.flatnonzero((train_n_pos < 50) & (val_n_pos > 0))
rare_ap = {}
for run in [PARENT_RUN, *(r for r, _ in NEW_RUNS.values())]:
    _, per_cat = multilabel_metrics(y["val"], probs[run]["val"], categories)
    rare_ap[run] = per_cat["ap"].to_numpy()[rare]
rare_df = pd.DataFrame(rare_ap, index=np.array(categories)[rare]).assign(n_train=train_n_pos[rare], n_val=val_n_pos[rare])
print(f"validation AP on the {len(rare)} rare categories (< 50 training positives):\n")
print(rare_df.round(2).to_string())
print("\nmean:", {run: round(float(v.mean()), 3) for run, v in rare_ap.items()})

validation AP on the 9 rare categories (< 50 training positives):

                            modernbert_v2_biasinit  modernbert_v5_biasinit_lr3e-5  modernbert_v6_biasinit_posweight  n_train  n_val
Affiliate License-Licensor                    0.27                           0.14                              0.17       31      7
Joint Ip Ownership                            0.55                           0.36                              0.56       38     22
Most Favored Nation                           0.03                           0.04                              0.05       27      7
No-Solicit Of Customers                       0.25                           0.42                              0.51       30      3
No-Solicit Of Employees                       0.86                           0.77                              0.80       40     12
Non-Disparagement                             0.18                           0.09                              0.25       24     11
Price Res

### Ensemble follow-up (only if earned on tune)

`ensemble_top2` averages the two runs with the best tune macro AP. If E2 or E3 beats v2 on tune macro AP, it would
enter the top 2 and change the ensemble, so that combination is scored here as a follow-up. Otherwise the current
best model stays as it is.

In [11]:
candidates = [run for run, _ in NEW_RUNS.values()
              if retrain_df.query("model == @run and thresholds == 'best F1'")["tune_macro_ap"].iloc[0]
              > retrain_df.query("model == @PARENT_RUN and thresholds == 'best F1'")["tune_macro_ap"].iloc[0]]
follow_up_rows = []
if not candidates:
    print("neither E2 nor E3 beats v2 on tune macro AP, so ensemble_top2 stays the best model, no follow-up needed")
else:
    all_runs = [*PARENT_RUNS, *candidates]
    tune_ap = {run: multilabel_metrics(y["tune"], probs[run]["tune"], categories)[0]["macro_ap_present"] for run in all_runs}
    new_top2 = sorted(all_runs, key=tune_ap.get, reverse=True)[:2]
    probs["ensemble_top2_new"] = {split: np.mean([probs[run][split] for run in new_top2], axis=0) for split in ("tune", "val")}
    thr = pick_thresholds(y["tune"], probs["ensemble_top2_new"]["tune"])
    follow_up_rows.append(log_row("E2/E3 follow-up", "E0", "ensemble_top2_new",
                                  f"mean probability of {' + '.join(new_top2)}, best-F1 cutoffs", thr))
    print(pd.DataFrame(follow_up_rows).drop(columns="what").round(3).to_string(index=False))

  experiment_id parent             model  tune_macro_ap  tune_macro_f1  val_macro_ap  val_micro_precision  val_micro_recall  val_micro_f1  val_macro_f1
E2/E3 follow-up     E0 ensemble_top2_new          0.588          0.607         0.606                0.598             0.698         0.644         0.543


In [12]:
# experiment log for #24: E0 is the current best model, E1-E3 each change one thing; v2 is logged as E2/E3's parent
v2_thr = pick_thresholds(y["tune"], probs[PARENT_RUN]["tune"])
log_rows = [
    {**parent_row, "thresholds": "best F1"},
    {**e1_row, "thresholds": f"best F1, floor N={best_floor}"},
    {**log_row("v2", "", PARENT_RUN, "parent of E2/E3: modernbert_v2_biasinit (lr 5e-5, bias init)", v2_thr),
     "thresholds": "best F1"},
]
for exp_id, (run, what) in NEW_RUNS.items():
    log_rows.append({**log_row(exp_id, "v2", run, what, pick_thresholds(y["tune"], probs[run]["tune"])),
                     "thresholds": "best F1"})
log_rows += [{**row, "thresholds": "best F1"} for row in follow_up_rows]

experiment_log = pd.DataFrame(log_rows)
experiment_log = experiment_log[["experiment_id", "parent", "model", "what", "thresholds",
                                 *[c for c in experiment_log.columns if c.startswith(("tune_", "val_"))]]]
experiment_log.to_csv(OUT_DIR / "experiment_log.csv", index=False)
print(experiment_log.drop(columns="what").round(3).to_string(index=False))

  experiment_id parent                            model         thresholds  tune_macro_ap  tune_macro_f1  val_macro_ap  val_micro_precision  val_micro_recall  val_micro_f1  val_macro_f1
             E0                           ensemble_top2            best F1          0.590          0.604         0.593                0.633             0.698         0.664         0.559
             E1     E0                    ensemble_top2 best F1, floor N=1          0.590          0.604         0.593                0.633             0.698         0.664         0.559
             v2                  modernbert_v2_biasinit            best F1          0.569          0.598         0.584                0.594             0.693         0.640         0.536
             E2     v2    modernbert_v5_biasinit_lr3e-5            best F1          0.508          0.541         0.528                0.555             0.672         0.608         0.492
             E3     v2 modernbert_v6_biasinit_posweight            bes

## Note: how much of the confusion would clause grouping remove?

Arielle's unmerged `clause-grouping` branch (commit `44e1d4a`, `notebooks/cuad_grouping`) maps the 41 categories into
12 groups and drops Document Name and Parties. Every experiment here stays on all 41, per the project brief. This cell
only measures how many of the current best model's validation false alarms are confusions *inside* one of her groups.
Those would stop counting as errors if the team switched to grouped labels. The mapping is copied here because the
branch isn't merged.

In [13]:
# copied from origin/clause-grouping (44e1d4a) notebooks/cuad_grouping; not merged, so not importable from main
CLAUSE_GROUPS = {
    "Dates & Term": ["Agreement Date", "Effective Date", "Expiration Date", "Renewal Term", "Notice Period To Terminate Renewal"],
    "Competitive Restrictions": ["Non-Compete", "Exclusivity", "No-Solicit Of Customers", "Competitive Restriction Exception"],
    "Control & Assignment": ["Change Of Control", "Anti-Assignment"],
    "Licensing": ["License Grant", "Non-Transferable License", "Affiliate License-Licensor", "Affiliate License-Licensee",
                  "Unlimited/All-You-Can-Eat-License", "Irrevocable Or Perpetual License"],
    "Post-Termination & Audit": ["Post-Termination Services", "Audit Rights"],
    "Liability": ["Uncapped Liability", "Cap On Liability"],
    "Financial & Commercial Terms": ["Revenue/Profit Sharing", "Minimum Commitment", "Price Restrictions", "Most Favored Nation",
                                     "Volume Restriction", "Rofr/Rofo/Rofn"],
    "Intellectual Property & Technology": ["Ip Ownership Assignment", "Joint Ip Ownership", "Source Code Escrow", "Covenant Not To Sue"],
    "Termination for Convenience": ["Termination For Convenience"],
    "Legal & Contractual Protections": ["Governing Law", "Insurance", "Third Party Beneficiary", "Warranty Duration"],
    "Liquidated Damages": ["Liquidated Damages"],
    "Employee & Business Conduct": ["No-Solicit Of Employees", "Non-Disparagement"],
}
EXCLUDED_CATEGORIES = {"Document Name", "Parties"}
group_of = {cat: group for group, cats in CLAUSE_GROUPS.items() for cat in cats}
assert set(group_of) | EXCLUDED_CATEGORIES == set(categories), "grouping doesn't cover the 41 categories"

# false alarms of the current best model (E0) on validation
best_pred = probs["ensemble_top2"]["val"] >= parent_thr
r, c = np.nonzero(best_pred & (y["val"] == 0))
cat_names = np.array(categories)
true_cats = [set(cat_names[y["val"][row] == 1]) for row in r]


def fp_kind(pred_cat, chunk_cats):
    if pred_cat in EXCLUDED_CATEGORIES:
        return "predicted category dropped by grouping"
    if not chunk_cats:
        return "no clause in chunk"
    if any(group_of.get(t) == group_of[pred_cat] for t in chunk_cats):
        return "same group as a clause in the chunk"
    return "different group"


grouping_fp = pd.DataFrame({"category": cat_names[c], "kind": [fp_kind(p, t) for p, t in zip(cat_names[c], true_cats)]})
grouping_fp.to_csv(OUT_DIR / "grouping_fp_breakdown.csv", index=False)
print(f"{len(grouping_fp)} validation false positives (current best model):\n")
print(grouping_fp["kind"].value_counts().to_string(), "\n")
print("biggest within-group confusions by predicted category:")
print(grouping_fp[grouping_fp["kind"] == "same group as a clause in the chunk"]["category"].value_counts().head(8).to_string())

684 validation false positives (current best model):

kind
same group as a clause in the chunk       261
different group                           211
no clause in chunk                        188
predicted category dropped by grouping     24 

biggest within-group confusions by predicted category:
category
Affiliate License-Licensor    63
Uncapped Liability            27
Volume Restriction            19
Affiliate License-Licensee    19
Effective Date                17
Most Favored Nation           13
Renewal Term                  11
Price Restrictions            11


## Findings (validation set)

**Current best model is unchanged:** `ensemble_top2` (mean of `modernbert_v2_biasinit` + `modernbert_v1_6ep`) with
per-category best-F1 cutoffs. Validation macro F1 0.559, micro F1 0.664, macro AP 0.593. All choices were made on tune.

| ID | Change | Parent | Result |
|---|---|---|---|
| E1 | Threshold floor: categories with fewer than N tune positives keep 0.5 | E0 | Didn't help. Cross-validation on tune picks N = 1 (no floor). |
| E2 | Learning rate 3e-5 instead of 5e-5 | v2 | Hurt. Val macro AP 0.584 to 0.528 (difference -0.056, 95% CI -0.081 to -0.037). |
| E3 | Class weights (`pos_weight=sqrt`) on top of bias init | v2 | Neutral overall, better on rare categories. Val macro AP +0.002 (CI -0.023 to +0.029). |

**What helped**

- **E3 lifts the rare categories.** Mean validation AP on the 9 categories with fewer than 50 training positives rises
  from 0.33 to 0.37. Price Restrictions goes 0.05 to 0.21, Source Code Escrow 0.13 to 0.30, No-Solicit Of Customers
  0.25 to 0.51. Third Party Beneficiary drops (0.63 to 0.46), and micro AP is slightly lower (-0.017), so common
  categories pay a little.
- **E3 needs less threshold tuning.** At the plain 0.5 cutoff it has the best macro F1 of any single run (0.545 vs
  0.506 for v2) and the highest recall (0.729). Class weights push its probabilities up, so more rare-category
  chunks clear 0.5.

**What didn't help**

- **E1 threshold floor.** On the macro F1 rule set up front, no floor wins. A floor of N = 3 is noted as a precision
  option (seen after validation, not selected): it gives Affiliate License-Licensor and Most Favored Nation the 0.5
  cutoff, which cuts validation false positives from 684 to 506 at flat macro F1, but drops their recall.
- **E2 lower learning rate.** Clearly worse in 3 epochs, consistent with #28's finding that v2 was still improving at
  epoch 3. A lower rate would need more epochs to catch up.
- **E3 in the ensemble.** E3 edges v2 on tune macro AP (0.570 vs 0.569), so E3 + v2 was tried as a new top 2. Its
  tune macro AP (0.588) is below the current ensemble (0.590), so it isn't selected.

**Clause grouping (note only, not modeled).** Of the current best model's 684 validation false positives, 261 are
confusions between categories in the same group of Arielle's 12-group mapping (63 of them Affiliate License-Licensor
on licensing chunks), and 24 are predictions of Document Name or Parties, which the mapping drops. About 42% of false
alarms would stop counting as errors under grouped labels. That's an upper bound, because grouping also changes
which chunks are positive.

**Caveats.** Each setup was trained once with one seed. The intervals cover which contracts are in validation, not
training randomness. E2 and E3 used transformers 5.21.0 where v2 used 5.20.1; everything else in the setup matches.

**Suggested next experiments.** (GPU) v2 + class weights for 5 or 6 epochs, since class weights helped rare categories
and v2 was still improving at epoch 3. (No GPU) average E3 with `ensemble_top2` and pick per-category cutoffs, or use
E3's 0.5 cutoffs for rare categories only.